# PerGAZE GazeformerISP ? Kaggle

Enable **Internet**, attach the PerGAZE dataset, and select **GPU T4 x2**
(or one GPU). Public GitHub access requires no token. Two available GPUs use
DataParallel and global batch 2; on one GPU the notebook uses global batch 1.
This is a hardware batch override; other training defaults directly match the
AiR training hyperparameters with the requested 5+5 schedule: 10 epochs, RL at index 5, seed 10, 6 encoder/6 decoder layers,
hidden dim 512, max length 16, one-epoch warmup and linear LR decay.
Images: 1024x768; scanpath frame: 512x352.
Train: train.json; validation: test_seen.json. support.json/test_unseen.json
are not consumed by this pipeline. Learned text embeddings are the default.
Optional SentenceTransformer text embeddings are provided below.


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE = Path("/kaggle/working/IndividualScanpath")
if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ntnghia06/IndividualScanpath.git", str(CODE)], check=True)
SRC = CODE / "PerGAZE/GazeformerISP/src"
assert (SRC / "parallel.py").is_file(), "Update the repository for Gazeformer DataParallel support"


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze sentence-transformers


Kaggle provides torch/torchvision. Check their import compatibility before training.


In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
assert torch.cuda.is_available(), "Enable GPU in Kaggle Settings"
GPU_IDS = list(range(min(2, torch.cuda.device_count())))
for i in GPU_IDS:
    print(i, torch.cuda.get_device_name(i))
TRAIN_BATCH = len(GPU_IDS)

DATA = Path("/kaggle/input/datasets/ngtrongnghia/pergaze/dataset")
RUN = Path("/kaggle/working/pergaze_gazeformer_run_original_rl_fp32")
for name in ("train.json", "test_seen.json", "images"):
    assert (DATA / name).exists(), f"Missing: {DATA / name}"
COMMON = [
    "--train_file", str(DATA / "train.json"),
    "--val_file", str(DATA / "test_seen.json"),
    "--img_dir", str(DATA / "images"),
    "--att_dir", str(DATA / "attention_reasoning"),
    "--device", "cuda:0", "--gpu_ids", *map(str, GPU_IDS),
]
def run_script(script, *arguments):
    subprocess.run([sys.executable, "-u", str(SRC / script), *COMMON, *arguments], check=True)
print("GPU IDs:", GPU_IDS, "global training batch:", TRAIN_BATCH)


### Precompute frozen COCO image features and sentence embeddings
Run once before training. Existing valid image files are reused. FP32 features use tensor resize; older caches must be re-extracted.


In [ ]:
CACHE_ROOT = Path("/kaggle/working/pergaze_cache_fp32_v2")
FEATURE_DIR = CACHE_ROOT / "image_features"
TEXT_FILE = CACHE_ROOT / "gazeformer_task_embeddings.npz"
# All four splits: frozen encoders use images/task text, never scanpath labels.
FILES = [DATA / name for name in ("train.json", "test_seen.json", "support.json", "test_unseen.json")
         if (DATA / name).is_file()]
subprocess.run([
    sys.executable, "-u", str(SRC / "preprocess/feature_extractor.py"),
    "--files", *map(str, FILES), "--img_dir", str(DATA / "images"),
    "--feature_dir", str(FEATURE_DIR), "--output", str(TEXT_FILE),
    "--device", "cuda:0", "--gpu_ids", *map(str, GPU_IDS),
    "--batch", "4", "--workers", "2",
], check=True)
COMMON.extend(["--feature_dir", str(FEATURE_DIR), "--text_embeddings", str(TEXT_FILE)])


Validate every JSON record, image and required bbox/attention map. No training occurs here.


In [ ]:
run_script("preprocess/validate_dataset.py", "--log_root", str(RUN))


Bounded smoke check on real samples from all three conditions. Duplicated
samples form a batch large enough for both GPUs. Uses a small random model
only for forward/backward/inference/RL checks; full training below restores
the default 6-layer, hidden-dim-512 architecture and pretrained backbone.


In [ ]:
run_script("train.py", "--smoke_test", "--batch", str(TRAIN_BATCH),
           "--max_length", "3", "--hidden_dim", "128", "--num_encoder", "1",
           "--num_decoder", "1", "--rl_sample_number", "2",
           "--log_root", "/kaggle/working/pergaze_gazeformer_smoke")


Full training. The COCO Mask R-CNN ResNet50 backbone downloads on first use
and is frozen by default. Features are extracted online, with no image-feature
preprocessing step. Epoch indices 0-4 are supervised (SFT) and 5-9 use RL.
Both models validate after every epoch. The one-epoch warmup is within SFT;
supervised.pth is saved after displayed epoch 5.
Train and validation have text progress bars when active. Run this cell for
a fresh run; use the resume cell instead if a checkpoint already exists.


In [ ]:
run_script("train.py", "--log_root", str(RUN),
           "--batch", str(TRAIN_BATCH), "--workers", "2")


Optional resume: run instead of the fresh-training cell. Restore the entire
run directory first after a new Kaggle session. Training batch and hyperparameters
must match the saved run; the number of GPUs can change. If the saved run used
batch 2, keep TRAIN_BATCH=2 even when resuming with one GPU. `--epoch N` sets a
total target, not N extra epochs.


In [ ]:
# Uncomment to resume:
# run_script("train.py", "--resume", "--log_root", str(RUN),
#            "--batch", str(TRAIN_BATCH), "--workers", "2")


Evaluate best.pth on test_seen.json. This is validation evaluation, not an independent unseen test. Default evaluation batch 1 matches AiR; pass --test_batch 2 explicitly to distribute larger evaluation batches across two GPUs.


In [ ]:
run_script("test.py", "--log_root", str(RUN), "--workers", "2")


In [ ]:
import json
report = json.loads((RUN / "evaluation_validation.json").read_text())
print(json.dumps(report["overall"], indent=2))


Save notebook output/checkpoints before ending the session. When precomputed semantic text is enabled, the embedding archive is also stored in /kaggle/working and must be retained for resume.


In [ ]:
import shutil
output = shutil.make_archive("/kaggle/working/pergaze_gazeformer_results", "zip",
                             root_dir=RUN.parent, base_dir=RUN.name)
print(output)
print("Retain for resume/few-shot:", TEXT_FILE, FEATURE_DIR)
